# LOGAN: discrimination within ORF-length bins

**Draft caption.** ROC AUC of out-of-fold model scores evaluated within ORF-length bins, for complete Prodigal+ ORFs against all complete Prodigal− ORFs (a) and against intergenic ORFs (b). Within a bin, length carries little information, so the length-only model falls towards 0.5 (dotted) from 75 aa upward while models including 3Di entropy remain high. Bins with fewer than 200 sampled positives or negatives are omitted. Same models and cross-validation as the beyond-length figure.

**Outputs:** `../images/figL03_models_within_length.pdf` (vector) and `../images/figL03_models_within_length.png` (300 dpi).

**Data** (plotting-ready, committed; produced by `../prep/logan_prepare_data.py` on Setonix):

- `../data/figL03_models_by_length.tsv.gz` (217 rows; sha256 `241f3c61a0a6…`): within-bin AUC of out-of-fold scores; sampled_pos/neg counted from 90 sample shards. Source on Setonix (`$LOGAN_ROOT`): `qc/analysis_models_final/models_by_length.tsv`.

**Upstream.** LOGAN v1.2 metagenomes (50,009), genome_entropy #102 (closed by PR #115); analysis scripts `Validation/logan/scripts/` at commit `06080b4`. Full run provenance: `../data/logan_provenance.json`.

**Run:** `jupyter nbconvert --to notebook --execute figL03_models_within_length.ipynb` from `manuscript/notebooks/`. Needs only pandas, numpy, matplotlib and seaborn; no HPC or remote paths.

In [ ]:
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

DATA, IMAGES = Path("../data"), Path("../images")
IMAGES.mkdir(exist_ok=True)
plt.rcParams["svg.hashsalt"] = "genome_entropy"   # deterministic SVG ids if SVG is ever written

# Colour slots (fixed order, colour-blind-checked categorical palette used across the LOGAN figures)
SLOTS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7"]
INK, INK2, GRID, SURFACE = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"
sns.set_theme(context="paper", style="ticks", font_scale=1.15, rc={
    "axes.edgecolor": INK2, "axes.labelcolor": INK, "xtick.color": INK2, "ytick.color": INK2,
    "text.color": INK, "axes.facecolor": SURFACE, "figure.facecolor": "white", "grid.color": GRID,
    "axes.grid": True, "grid.linewidth": 0.6, "axes.spines.top": False, "axes.spines.right": False,
    "legend.frameon": False, "lines.linewidth": 2, "font.family": "DejaVu Sans"})


def panel(ax, letter, x=-0.08, y=1.04):
    ax.text(x, y, letter, transform=ax.transAxes, fontsize=15, fontweight="bold", va="bottom", ha="right")


def save(fig, name):
    """Vector PDF (fixed metadata, so reruns are identical) and a 300-dpi PNG preview."""
    fig.savefig(IMAGES / f"{name}.pdf", bbox_inches="tight", metadata={"CreationDate": None, "ModDate": None})
    fig.savefig(IMAGES / f"{name}.png", dpi=300, bbox_inches="tight", metadata={"Software": None})
    plt.close(fig)
    print("wrote", IMAGES / f"{name}.pdf", IMAGES / f"{name}.png")

In [ ]:
r = pd.read_csv(DATA / "figL03_models_by_length.tsv.gz", sep="\t")
MIN_N = 200
r = r[(r.sampled_pos >= MIN_N) & (r.sampled_neg >= MIN_N)]
SETS = {"length": "length", "length+aa": "+ AA entropy", "length+3di": "+ 3Di entropy",
        "length+mi": "+ 3Di–12st MI", "length+aa+3di+12st+mi": "+ all four"}
fig, axes = plt.subplots(1, 2, figsize=(15, 5.4), sharey=True)
for ax, (contrast, ctxt), letter in zip(axes, [("vs all Prodigal-", "all Prodigal−"),
                                                ("vs intergenic", "intergenic")], "ab"):
    d = r[r.contrast == contrast]
    order = sorted(d.length_bin.unique(), key=lambda s: float(s.strip("[(").split(",")[0]))
    labels = [f"{int(float(s.strip('[(').split(',')[0]))}–" for s in order]
    for i, (s, lab) in enumerate(SETS.items()):
        x = d[d.features == s].set_index("length_bin").reindex(order)
        ax.plot(range(len(order)), x.roc_auc, marker="o", ms=8, color=SLOTS[i], label=lab)
    ax.set_xticks(range(len(order)), labels)
    ax.set_xlabel("ORF length bin (aa)")
    ax.set_ylabel("ROC AUC within bin")
    ax.set_title(f"Prodigal+ vs {ctxt}", loc="left", fontsize=11)
    ax.axhline(0.5, color=INK2, lw=1, ls=":")
    panel(ax, letter)
h, l = axes[0].get_legend_handles_labels()
fig.legend(h, l, loc="lower center", ncol=5, bbox_to_anchor=(0.5, -0.02), fontsize=10)
fig.tight_layout(rect=(0, 0.06, 1, 1))
save(fig, "figL03_models_within_length")